In [1]:
import pandas as pd
import numpy as np
from pandas import DataFrame, Series
import sys          # only needed to determine Python version number

# Set some pandas parameters..
#pd.set_option('display.notebook_repr_html', True)
#pd.set_option('display.max_columns', 10)
#pd.set_option('display.max_rows', 10)

# Matplotlib ptions
import matplotlib.pyplot as plt
import matplotlib   # only needed to determine Matplotlib version number
# Render our plots inline:
%matplotlib inline  
pd.set_option('display.mpl_style', 'default') # Make the graphs a bit prettier
plt.rcParams['figure.figsize'] = (15, 5)

print('Python version ' + sys.version)
print('Pandas version ' + pd.__version__)
#print('Matplotlib version ' + matplotlib.__version__)
#pd.show_versions()


Python version 3.4.3 (default, Oct 14 2015, 20:28:29) 
[GCC 4.8.4]
Pandas version 0.17.1


In [2]:
#scenarios = ['P0-Mi-Ei'] # for quick testing
scenarios = ['P0-Mh-Ei', 'P1-Mh-Ei', 'P0-Mi-Ei', 'P1-Mi-Ei', 'P0-Me-Ei', 'P1-Me-Ei', 'P0-Mi-E0', 'P1-Mi-E0', 'P0-Mi-E1', 'P1-Mi-E1']
baseIDir = 'output/results'

productData_byscenario = [pd.read_csv(baseIDir+'/productData_'+scenario+'.csv',sep=';',  dtype={'value': np.float64}) for scenario in scenarios]
productDataReg = pd.concat(productData_byscenario)
productDataReg = productDataReg.drop(['Unnamed: 8'],axis=1) # we don't want this column never
#productDataReg['prodLev'] = np.where(productDataReg['prod'] in ['hardWRoundW', 'softWRoundW', 'pulpWFuelW'], 1, 2)
productDataReg['prodLev'] = np.where(productDataReg['prod'].isin(['hardWRoundW', 'softWRoundW', 'pulpWFuelW','ashRoundW']), 'primPr', 'secPr')
productDataReg = productDataReg.drop(['freeDim'],axis=1) # we don't want this column never
productDataReg = productDataReg[productDataReg['parName'] != 'rt'] # we don't want rt in national data
productDataReg.set_index(['scen','country','region','prod','prodLev','year','parName'], inplace=True)
productDataReg=productDataReg.unstack(['parName'])

lagged_productDataReg = productDataReg.groupby(level=['scen','country','region','prod','prodLev']).shift()




productDataReg[90:110]







value                      \
parName                                               dl        da        dt   
scen     country region prod      prodLev year                                 
P0-Me-Ei 11000   11010  ashPlyW   secPr   2095  0.005718  0.007879  0.013597   
                                          2096  0.005718  0.007879  0.013597   
                                          2097  0.005718  0.007879  0.013597   
                                          2098  0.005718  0.007879  0.013597   
                                          2099  0.005718  0.007879  0.013597   
                                          2100  0.005718  0.007879  0.013597   
                                          2101  0.005718  0.007879  0.013597   
                                          2102  0.005719  0.007879  0.013597   
                                          2103  0.005719  0.007879  0.013598   
                                          2104  0.005719  0.007879  0.013598   
                                          2105  0.005720  0.007878  0.013598   
                                          2106  0.005719  0.007878  0.013598   
                                          2107  0.005719  0.007878  0.013598   
                        ashRoundW primPr  2005  0.000303  0.000000  0.000303   
                                          2006  0.000303  0.000000  0.000303   
                                          2007  0.000534  0.000000  0.000534   
                                          2008  0.000587  0.000000  0.000587   
                                          2009  0.000646  0.000000  0.000646   
                                          2010  0.000711  0.000000  0.000711   
                                          2011  0.000782  0.000000  0.000782   

                                                                              \
parName                                               dc        sl        sa   
scen     country region prod      prodLev year                                 
P0-Me-Ei 11000   11010  ashPlyW   secPr   2095  0.007122  0.000000  0.000000   
                                          2096  0.007122  0.000000  0.000000   
                                          2097  0.007122  0.000000  0.000000   
                                          2098  0.007122  0.000000  0.000000   
                                          2099  0.007122  0.000000  0.000000   
                                          2100  0.007122  0.000000  0.000000   
                                          2101  0.007122  0.000000  0.000000   
                                          2102  0.007122  0.000000  0.000000   
                                          2103  0.007122  0.000000  0.000000   
                                          2104  0.007122  0.000000  0.000000   
                                          2105  0.007122  0.000000  0.000000   
                                          2106  0.007122  0.000000  0.000000   
                                          2107  0.007122  0.000000  0.000000   
                        ashRoundW primPr  2005  0.000000  0.012059  0.002225   
                                          2006  0.000000  0.012059  0.002225   
                                          2007  0.000000  0.011260  0.002396   
                                          2008  0.000000  0.011339  0.002453   
                                          2009  0.000000  0.011180  0.001915   
                                          2010  0.000000  0.011121  0.002019   
                                          2011  0.000000  0.011095  0.002273   

                                                                              \
parName                                               st        sc        pl   
scen     country region prod      prodLev year                                 
P0-Me-Ei 11000   11010  ashPlyW   secPr   2095  0.000000  0.000000  607.8520   
                                          2096  0

## Computing producer and consumer surpluses ##

### Producer surplus ###
$ S_t = S_{t-1} P_{t-1}^{-\sigma} * {\left(  \frac{In_t}{In_{t-1}} \right)}^ {\frac{\gamma}{\sigma} } * P_{t}^{\sigma}  $

$ P_t = P _{t-1} * S_{t-1}^{-\frac{1}{\sigma}}* {\left(  \frac{In_{t-1}}{In_t} \right)}^ {\frac{\gamma}{\sigma} } * S_t^{\frac{1}{\sigma}} $

$ \int P(S) ~ dS = P _{t-1} * S_{t-1}^{-\frac{1}{\sigma}}* {\left(  \frac{In_{t-1}}{In_t} \right)}^ {\frac{\gamma}{\sigma} } * S_{t}^{\frac{\sigma+1}{\sigma}} *  \frac{\sigma}{\sigma+1} $

Producer surplus =  $ P^* * S^* - P _{t-1} * S_{t-1}^{-\frac{1}{\sigma}}* {\left(  \frac{In_{t-1}}{In_t} \right)}^ {\frac{\gamma}{\sigma} } * S^{* ~ \frac{\sigma+1}{\sigma}} *  \frac{\sigma}{\sigma+1}    $

### Consumer surplus ###

$ D_t = D_{t-1} * P_{t-1}^{-\sigma}*P_t^\sigma$

$P_t = P_{t-1} * D_{t-1}^{-\frac{1}{\sigma}} * D_t^\frac{1}{\sigma} $

$ \int P(D) ~ dD = P_{t-1} * D_{t-1}^{-\frac{1}{\sigma}} * D_t^\frac{\sigma+1}{\sigma} * \frac{\sigma}{\sigma +1 }$

Consumer surplus = $ \left[ P_{t-1} * D_{t-1}^{-\frac{1}{\sigma}} * \frac{\sigma}{\sigma +1 } \right] \left[  D^{*~\frac{\sigma+1}{\sigma}} - (\tau * D^*)^ \frac{\sigma+1}{\sigma} \right] - (1- \tau)*D_t^* * P_t^*$

### Balance equation ###
PROD_forLocal (sl) + PROD_forExp (sa) + IMP (da) + sum_reg(reg_trade_in) = CONS_fromLocal (dl) + CONS_fromImp (da) + EXP (sa) + sum_reg(reg_trade_out)

(Note that this means that sl includes alread reg_trade_out, and dl includes already reg_trade_in)


In [3]:
# Computing 
gamma = 0.5
trs   = 0.1


# supplies and demands in Mm^3, prices in €/m^3
sigma =        productDataReg['value','sigma']
pc    =        productDataReg['value','pc']
pc_l  = lagged_productDataReg['value','pc']
sc    =        productDataReg['value','sc']*1000000
sc_l  = lagged_productDataReg['value','sc']*1000000
dc    =        productDataReg['value','dc']*1000000
dc_l  = lagged_productDataReg['value','dc']*1000000
inv   =        productDataReg['value','in']*1000000
inv_l = lagged_productDataReg['value','in']*1000000


productDataReg['value','surpl_pr']  =  (sc * pc) - \
                                      (pc_l /  pow(sc_l,1/sigma)) \
                                    * pow(inv_l/inv,gamma/sigma) \
                                    * pow(sc,(1+sigma)/sigma) \
                                    * (sigma / (1+sigma))

productDataReg['value','surpl_co']  =    (pc_l*pow(dc_l,-1/sigma)*sigma/(sigma+1)) \
                                      * (pow(dc,(sigma+1)/sigma)-pow(trs*dc,(sigma+1)/sigma)) \
                                      - (1-trs) * dc * pc
        
productDataReg['value','surpl_tot'] =  productDataReg.fillna(0)['value','surpl_pr']+ productDataReg.fillna(0)['value','surpl_co']        
        
productDataReg[90:110]


value                      \
parName                                               dl        da        dt   
scen     country region prod      prodLev year                                 
P0-Me-Ei 11000   11010  ashPlyW   secPr   2095  0.005718  0.007879  0.013597   
                                          2096  0.005718  0.007879  0.013597   
                                          2097  0.005718  0.007879  0.013597   
                                          2098  0.005718  0.007879  0.013597   
                                          2099  0.005718  0.007879  0.013597   
                                          2100  0.005718  0.007879  0.013597   
                                          2101  0.005718  0.007879  0.013597   
                                          2102  0.005719  0.007879  0.013597   
                                          2103  0.005719  0.007879  0.013598   
                                          2104  0.005719  0.007879  0.013598   
                                          2105  0.005720  0.007878  0.013598   
                                          2106  0.005719  0.007878  0.013598   
                                          2107  0.005719  0.007878  0.013598   
                        ashRoundW primPr  2005  0.000303  0.000000  0.000303   
                                          2006  0.000303  0.000000  0.000303   
                                          2007  0.000534  0.000000  0.000534   
                                          2008  0.000587  0.000000  0.000587   
                                          2009  0.000646  0.000000  0.000646   
                                          2010  0.000711  0.000000  0.000711   
                                          2011  0.000782  0.000000  0.000782   

                                                                              \
parName                                               dc        sl        sa   
scen     country region prod      prodLev year                                 
P0-Me-Ei 11000   11010  ashPlyW   secPr   2095  0.007122  0.000000  0.000000   
                                          2096  0.007122  0.000000  0.000000   
                                          2097  0.007122  0.000000  0.000000   
                                          2098  0.007122  0.000000  0.000000   
                                          2099  0.007122  0.000000  0.000000   
                                          2100  0.007122  0.000000  0.000000   
                                          2101  0.007122  0.000000  0.000000   
                                          2102  0.007122  0.000000  0.000000   
                                          2103  0.007122  0.000000  0.000000   
                                          2104  0.007122  0.000000  0.000000   
                                          2105  0.007122  0.000000  0.000000   
                                          2106  0.007122  0.000000  0.000000   
                                          2107  0.007122  0.000000  0.000000   
                        ashRoundW primPr  2005  0.000000  0.012059  0.002225   
                                          2006  0.000000  0.012059  0.002225   
                                          2007  0.000000  0.011260  0.002396   
                                          2008  0.000000  0.011339  0.002453   
                                          2009  0.000000  0.011180  0.001915   
                                          2010  0.000000  0.011121  0.002019   
                                          2011  0.000000  0.011095  0.002273   

                                                                              \
parName                                               st        sc        pl   
scen     country region prod      prodLev year                                 
P0-Me-Ei 11000   11010  ashPlyW   secPr   2095  0.000000  0.000000  607.8520   
                                          2096  0

In [4]:
productDataReg_byprodLev = productDataReg.groupby(level=['scen','country','region','prodLev','year']).sum() # attenction, everything is just summed, prices are obviously not correct
productDataReg_byprodLev[:130]

#productDataReg['value']=999

#test
#print(productDataReg[:500])
#st_reg = productDataReg[productDataReg['parName']=='st']
#st_reg = st_reg.drop(['freeDim'],axis=1) 
#st_reg.rename(columns={'value':'st_value'}, inplace=True)
#st_reg.set_index(['scen','country','region','prod','year','parName'], inplace=True)
#result = pd.concat([productDataNat, st_reg], axis=1)
#productDataNat = productDataReg.drop(['freeDim'],axis=1)    # we don't want this in national data, but still keep it in regional one
#productDataNat = productDataNat[productDataNat['parName'] != 'rt'] # we don't want rt in national data
#productDataNat.set_index(['scen','country','region','prod','year','parName'], inplace=True)
#productDataNat[:5]

value                       \
parName                                      dl        da         dt   
scen     country region prodLev year                                   
P0-Me-Ei 11000   11010  primPr  2005   1.177444  0.000000   1.177444   
                                2006   1.177444  0.000000   1.177444   
                                2007   2.072304  0.000000   2.072304   
                                2008   2.279538  0.000000   2.279538   
                                2009   2.507488  0.000000   2.507488   
                                2010   2.756074  0.000000   2.756074   
                                2011   3.021288  0.000000   3.021288   
                                2012   3.103714  0.000000   3.103714   
                                2013   3.179768  0.000000   3.179768   
                                2014   3.190159  0.000000   3.190159   
                                2015   3.196638  0.000000   3.196638   
                                2016   3.203155  0.000000   3.203155   
                                2017   3.209848  0.000000   3.209848   
                                2018   3.208600  0.000000   3.208600   
                                2019   3.153010  0.000000   3.153010   
                                2020   3.159477  0.000000   3.159477   
                                2021   3.166179  0.000000   3.166179   
                                2022   3.173169  0.000000   3.173169   
                                2023   3.180419  0.000000   3.180419   
                                2024   3.187717  0.000000   3.187717   
                                2025   3.184859  0.000000   3.184859   
                                2026   3.183371  0.000000   3.183371   
                                2027   3.192849  0.000000   3.192849   
                                2028   3.202942  0.000000   3.202942   
                                2029   3.213278  0.000000   3.213278   
                                2030   3.224314  0.000000   3.224314   
                                2031   3.236552  0.000000   3.236552   
                                2032   3.240777  0.000000   3.240777   
                                2033   3.259108  0.000000   3.259108   
                                2034   3.269693  0.000000   3.269693   
...                                         ...       ...        ...   
                                2105   3.691224  0.000000   3.691224   
                                2106   3.697871  0.000000   3.697871   
                                2107   3.704465  0.000000   3.704465   
                        secPr   2005  11.564165  3.232008  14.796165   
                                2006  11.564165  3.232008  14.796165   
                                2007  10.374879  2.976229  13.351109   
                                2008  10.436689  2.921773  13.358464   
                                2009  10.272546  3.329898  13.602442   
                                2010  10.438047  3.025161  13.463216   
                                2011  10.500003  2.920545  13.420548   
                                2012  10.404578  3.154567  13.559137   
                                2013  10.443490  3.099941  13.543421   
                                2014  10.470532  3.098237  13.568767   
                                2015  10.498092  3.096619  13.594720   
                                2016  10.524809  3.095113  13.619935   
                                2017  10.550721  3.093722  13.644438   
                                2018  10.574002  3.092382  13.666376   
                                2019  10.583598  3.091137  13.674741   
                                2020  10.607077  3.090036  13.697109   
                                2021  10.629821  3.089031  13.718848   
                                2022  10.651872  3.088110  13.739973   
                                2023  10.673402  3.087197  13.760591   
          

## Saving dataframes to csv

In [36]:
productDataReg_byprodLev.to_csv("output/productDataReg_byprodLev.csv")

In [5]:
h5 = pd.HDFStore('output/test.h5')
h5['productDataReg'] = productDataReg
h5.close()

#h5bis = pd.HDFStore('output/test.h5')
#test = h5bis['productDataReg'] 
#test.head()